#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

Faiss is widely used for nearest neighbor search and vector indexing in machine learning and NLP workflows. It supports both exact search and approximate search with advanced indexes such as IVF, HNSW, and PQ, making it suitable for large-scale embedding retrieval.

Key benefits of Faiss:
- fast similarity search for high-dimensional embeddings
- scalable indexing for millions or billions of vectors
- GPU support for accelerated nearest neighbor search
- flexible index types for balancing speed, accuracy, and memory usage

In this notebook, we use Faiss as a vector store backend to build efficient semantic search applications and retrieve similar documents from embedding collections.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs=text_splitter.split_documents(documents)

/Users/bravalji/Documents/LANGCHAIN/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky is blue." \n\nOrganizations use transformer models for all types of sequence conversions, from speech recognition to machine translation and protein sequence analysis.\n\nWhy are transformers important?\nEarly deep learning models that focused extensively on natural language processing (NLP) tasks aimed at getting computers to understand and respond to natural human language. They guessed the next word in a sequence based on the previous 

In [4]:
embeddings=OllamaEmbeddings(model="gemma:2b")
db=FAISS.from_documents(docs, embeddings)
db

In [7]:
### Querying the vector store
query = "How Transformers makes suggestions?"
results = db.similarity_search(query)
results[0].page_content

'Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.

A Retriever is a standard interface in LangChain that takes a query string and returns relevant documents. Unlike direct vector store searches, retrievers abstract away the underlying search mechanism, enabling seamless integration with RAG chains, question-answering systems, and other NLP pipelines.

Benefits of using Retrievers:
- standardized interface for document retrieval across different vector store backends
- chainable with language models and other components in LangChain
- support for metadata filtering, reranking, and hybrid search strategies
- easier integration with retrieval-augmented generation (RAG) workflows

By converting Faiss into a Retriever, we can feed search results directly into prompts and language models for downstream tasks like document-based question answering and content summarization.

In [8]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

Using similarity_search_with_score provides several advantages:
- retrieve documents with their relevance scores to evaluate retrieval quality
- filter results by setting threshold scores to keep only highly relevant documents
- rank and sort results by confidence before passing to downstream tasks
- debug and tune the embedding model by analyzing distance distributions

The L2 distance score represents how far apart two embeddings are in vector space. A score of 0 means identical embeddings, while higher scores indicate lower similarity. By examining the score distribution across results, you can determine appropriate thresholds for your application and make informed decisions about result reliability and ranking.

In [9]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='d1ab8c9a-2f48-451e-b87a-242d72450419', metadata={'source': 'speech.txt'}, page_content='Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'),
  np.float32(2006.7297)),
 (Document(id='ff894145-3a5c-46e0-9bc9-a35931f89bc2', metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky is blue." \n\nOrganizations use transformer models for all types of sequence conversions, fr

In [10]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.6533568501472473,
 -1.385776400566101,
 -0.3855245113372803,
 1.7575433254241943,
 0.6390717625617981,
 2.1407973766326904,
 0.6878575086593628,
 -1.034628987312317,
 0.514642059803009,
 -0.5756098031997681,
 2.046017646789551,
 0.5832450985908508,
 -0.2729344964027405,
 1.6131113767623901,
 0.3047635853290558,
 -0.05712195113301277,
 5.626896381378174,
 -0.7206969261169434,
 -0.7528225183486938,
 1.3884423971176147,
 2.396371841430664,
 -0.2210923284292221,
 0.7833462357521057,
 1.26875901222229,
 -1.5733789205551147,
 -0.6860765218734741,
 0.8300080299377441,
 0.8700966238975525,
 -0.18644830584526062,
 -3.4097166061401367,
 -0.6440046429634094,
 1.0328853130340576,
 -0.2321307510137558,
 0.3212449550628662,
 0.029936250299215317,
 -0.5517175197601318,
 1.1862634420394897,
 -1.2878906726837158,
 0.9151259064674377,
 -2.0967016220092773,
 -0.5545393824577332,
 0.018075235188007355,
 1.2513699531555176,
 0.031016893684864044,
 1.079835295677185,
 0.5034242868423462,
 1.988949656486

In [12]:
doc_score=db.similarity_search_by_vector(embedding_vector)
doc_score

[Document(id='d1ab8c9a-2f48-451e-b87a-242d72450419', metadata={'source': 'speech.txt'}, page_content='Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'),
 Document(id='ff894145-3a5c-46e0-9bc9-a35931f89bc2', metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky is blue." \n\nOrganizations use transformer models for all types of sequence conversions, from speech recognition to mac

In [13]:
### Saving And Loading
db.save_local("faiss_index")

In [15]:
new_db=FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [16]:
docs

[Document(id='d1ab8c9a-2f48-451e-b87a-242d72450419', metadata={'source': 'speech.txt'}, page_content='Transformer models fundamentally changed NLP technologies by enabling models to handle such long-range dependencies in text. The following are more benefits of transformers.'),
 Document(id='ff894145-3a5c-46e0-9bc9-a35931f89bc2', metadata={'source': 'speech.txt'}, page_content='Transformers are a type of neural network architecture that transforms or changes an input sequence into an output sequence. They do this by learning context and tracking relationships between sequence components. For example, consider this input sequence: "What is the color of the sky?" The transformer model uses an internal mathematical representation that identifies the relevancy and relationship between the words color, sky, and blue. It uses that knowledge to generate the output: "The sky is blue." \n\nOrganizations use transformer models for all types of sequence conversions, from speech recognition to mac